<a href="https://colab.research.google.com/github/danielelisalde/InfoTec-Inteligencia-Artificial-Aplicada-con-Llama/blob/main/Hackathon1_UAGro_Orienta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ============================================
# BLOQUE 1: INSTALACIÓN Y CONFIGURACIÓN
# ============================================

!pip install -q transformers peft accelerate trl sentence-transformers \
    huggingface_hub datasets groq gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 10.2 MB/s eta 0:00:00


In [5]:
# ============================================
# BLOQUE 2: IMPORTACIONES Y SECRETOS
# ============================================

import os
import re
import unicodedata
import numpy as np
import torch

from google.colab import userdata
from huggingface_hub import login

from sentence_transformers import SentenceTransformer
from groq import Groq

# Reducir mensajes innecesarios de Transformers
from transformers.utils import logging
logging.set_verbosity_error()

# Leer secretos de Colab
HF_TOKEN = userdata.get("HF_TOKEN")
GROQ_API_KEY = userdata.get("API_groq")

if not HF_TOKEN:
    raise ValueError("No se encontró el secreto HF_TOKEN.")

if not GROQ_API_KEY:
    raise ValueError("No se encontró el secreto API_groq.")

# Iniciar sesión en Hugging Face
login(token=HF_TOKEN)

# Conectar con Groq
client = Groq(api_key=GROQ_API_KEY)

print("Hugging Face conectado correctamente.")
print("Groq conectado correctamente.")
print("Entorno listo.")

Hugging Face conectado correctamente.
Groq conectado correctamente.
Entorno listo.


In [2]:
# ============================================
# BLOQUE 3: CARGAR ARCHIVOS UAGro
# ============================================

from google.colab import files

archivos_subidos = files.upload()

print("\nArchivos cargados:")
for nombre in archivos_subidos:
    print("-", nombre)

Saving RAG_contactos_programas_Oferta_Educativa_2026-2027 (1).txt to RAG_contactos_programas_Oferta_Educativa_2026-2027 (1).txt
Saving RAG_Catalogo_Oferta_Educativa_2026-2027_limpio.txt to RAG_Catalogo_Oferta_Educativa_2026-2027_limpio.txt

Archivos cargados:
- RAG_contactos_programas_Oferta_Educativa_2026-2027 (1).txt
- RAG_Catalogo_Oferta_Educativa_2026-2027_limpio.txt


In [6]:
# ============================================
# BLOQUE 3.1: LEER LOS ARCHIVOS
# ============================================

archivo_catalogo = next(
    nombre for nombre in archivos_subidos
    if "Catalogo" in nombre
)

archivo_contactos = next(
    nombre for nombre in archivos_subidos
    if "contactos" in nombre.lower()
)

with open(archivo_catalogo, "r", encoding="utf-8-sig") as archivo:
    texto_catalogo = archivo.read()

with open(archivo_contactos, "r", encoding="utf-8-sig") as archivo:
    texto_contactos = archivo.read()

print("Catálogo leído correctamente.")
print("Contactos leídos correctamente.")
print("Caracteres del catálogo:", len(texto_catalogo))
print("Caracteres de contactos:", len(texto_contactos))

Catálogo leído correctamente.
Contactos leídos correctamente.
Caracteres del catálogo: 47338
Caracteres de contactos: 56485


In [7]:
# ============================================
# BLOQUE 4: SEPARAR REGISTROS
# ============================================

def separar_registros(texto):
    registros = texto.split("---")
    registros_limpios = []

    for registro in registros:
        registro = registro.replace("\ufeff", "").strip()

        if len(registro) > 50:
            registros_limpios.append(registro)

    return registros_limpios


documentos_catalogo = separar_registros(texto_catalogo)
documentos_contactos = separar_registros(texto_contactos)

documentos = documentos_catalogo + documentos_contactos

print("Registros del catálogo:", len(documentos_catalogo))
print("Registros de contactos:", len(documentos_contactos))
print("Total de documentos:", len(documentos))

Registros del catálogo: 89
Registros de contactos: 89
Total de documentos: 178


In [8]:
# ============================================
# BLOQUE 4.1: CREAR CONTEXTO LIMPIO
# ============================================

documentos_contexto = []

for documento in documentos:
    lineas_utiles = []

    for linea in documento.splitlines():
        linea_limpia = linea.strip()

        if linea_limpia.startswith("PALABRAS CLAVE:"):
            continue

        if linea_limpia.startswith("INTENCION:"):
            continue

        lineas_utiles.append(linea)

    contexto_limpio = "\n".join(lineas_utiles).strip()
    documentos_contexto.append(contexto_limpio)

print("Contextos limpios creados:", len(documentos_contexto))

Contextos limpios creados: 178


In [9]:
# ============================================
# BLOQUE 4.2: VERIFICAR REGISTROS
# ============================================

for i, documento in enumerate(documentos_contexto[:3], start=1):
    print(f"\n========== DOCUMENTO {i} ==========")
    print(documento)


========== DOCUMENTO 1 ==========
CARRERA: Licenciatura en Enfermería
AREA: Ciencias de la Salud
UNIDAD ACADEMICA: Escuela Superior de Enfermeria No. 01
NIVEL: Superior
MODALIDAD: Escolarizada
DURACION: 10 Semestres o 5 años
LOCALIDAD: Chilpancingo
REGION: Centro

========== DOCUMENTO 2 ==========
CARRERA: Licenciatura en Enfermería
AREA: Ciencias de la Salud
UNIDAD ACADEMICA: Facultad de Enfermería No. 2
NIVEL: Superior
MODALIDAD: Escolarizada
DURACION: 10 Semestres o 5 años
LOCALIDAD: Acapulco
REGION: Acapulco

========== DOCUMENTO 3 ==========
CARRERA: Licenciatura en Enfermería
AREA: Ciencias de la Salud
UNIDAD ACADEMICA: Escuela Superior de Enfermería No. 3 UAGro
NIVEL: Superior
MODALIDAD: Escolarizada
DURACION: 10 Semestres o 5 años
LOCALIDAD: Ometepec
REGION: Costa Chica


In [10]:
# ============================================
# BLOQUE 5: GENERAR EMBEDDINGS
# ============================================

modelo_embeddings = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

embeddings_documentos = modelo_embeddings.encode(
    documentos,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embeddings generados correctamente.")
print("Cantidad de documentos:", len(documentos))
print("Forma de los embeddings:", embeddings_documentos.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Embeddings generados correctamente.
Cantidad de documentos: 178
Forma de los embeddings: (178, 384)


In [11]:
# ============================================
# BLOQUE 6: DETECTAR LOCALIDADES
# ============================================

def normalizar_texto(texto):
    texto = texto.lower().strip()

    texto = unicodedata.normalize("NFD", texto)

    texto = "".join(
        caracter
        for caracter in texto
        if unicodedata.category(caracter) != "Mn"
    )

    return texto


def detectar_localidad(pregunta):
    pregunta_normalizada = normalizar_texto(pregunta)

    if "cuajinicuilapa" in pregunta_normalizada:
        return "cuajinicuilapa"

    if "chilpancingo" in pregunta_normalizada:
        return "chilpancingo"

    if "acapulco" in pregunta_normalizada:
        return "acapulco"

    if "taxco" in pregunta_normalizada or "texco" in pregunta_normalizada:
        return "taxco"

    if "zumpango" in pregunta_normalizada:
        return "zumpango"

    if "ometepec" in pregunta_normalizada:
        return "ometepec"

    if "tecpan" in pregunta_normalizada:
        return "tecpan"

    if (
        "ciudad altamirano" in pregunta_normalizada
        or "cd altamirano" in pregunta_normalizada
    ):
        return "ciudad altamirano"

    if "coyuca de catalan" in pregunta_normalizada:
        return "coyuca de catalan"

    if "iguala" in pregunta_normalizada:
        return "iguala"

    if "zihuatanejo" in pregunta_normalizada:
        return "zihuatanejo"

    if "tixtla" in pregunta_normalizada:
        return "tixtla"

    return None


print(detectar_localidad("¿Qué carreras ofrece la UAGro en Texco?"))
print(detectar_localidad("¿Dónde hay Veterinaria en Cuajinicuilapa?"))

taxco
cuajinicuilapa


In [12]:
# ============================================
# BLOQUE 6.1: BÚSQUEDA RAG FILTRADA
# ============================================

def buscar_fragmentos(pregunta, cantidad=5):

    localidad = detectar_localidad(pregunta)

    # Seleccionar documentos candidatos
    if localidad:
        indices_candidatos = []

        for indice, documento in enumerate(documentos):
            documento_normalizado = normalizar_texto(documento)

            if localidad == "taxco":
                coincide = "taxco" in documento_normalizado

            elif localidad == "ciudad altamirano":
                coincide = (
                    "altamirano" in documento_normalizado
                    or "cd. altamirano" in documento_normalizado
                )

            else:
                coincide = localidad in documento_normalizado

            if coincide:
                indices_candidatos.append(indice)

    else:
        indices_candidatos = list(range(len(documentos)))

    # Si no hay documentos para la localidad, no inventar resultados
    if not indices_candidatos:
        return []

    # Vector de la pregunta
    embedding_pregunta = modelo_embeddings.encode(
        [pregunta],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    # Embeddings solo de los documentos candidatos
    embeddings_candidatos = embeddings_documentos[
        indices_candidatos
    ]

    similitudes = np.dot(
        embeddings_candidatos,
        embedding_pregunta
    )

    posiciones = np.argsort(similitudes)[::-1][:cantidad]

    resultados = []

    for posicion in posiciones:
        indice_real = indices_candidatos[posicion]

        resultados.append({
            "similitud": float(similitudes[posicion]),
            "documento": documentos_contexto[indice_real]
        })

    return resultados

In [13]:
# ============================================
# BLOQUE 6.2: PRUEBA TAXCO
# ============================================

resultados_taxco = buscar_fragmentos(
    "¿Qué carreras ofrece la UAGro en Taxco?",
    cantidad=3
)

for i, resultado in enumerate(resultados_taxco, start=1):
    print(f"\n--- Resultado {i} ---")
    print("Similitud:", round(resultado["similitud"], 3))
    print(resultado["documento"])


--- Resultado 1 ---
Similitud: 0.572
UNIDAD ACADEMICA: Centro Regional de Educación Superior Taxco el Viejo Campus Zona Norte
PROGRAMA: Licenciatura en Partería Profesional (Mixta)
LOCALIDAD: Taxco el viejo
REGION: Norte
DIRECTOR: MC. María Guadalupe Arroyo Rivera
DIRECCION: Carretera federal Taxco - Iguala, CP. 40323 Taxco el Viejo, Gro.
TELEFONO: Tel. 747 471 93 10 Ext. 3629, 3628 (número central UAGro)
CORREO: cres_zn@uagro.mx
PAGINA WEB: www.cres-zn.uagro.mx
FACEBOOK: https://www.facebook.com/campuszonanorte2018

--- Resultado 2 ---
Similitud: 0.564
CARRERA: Licenciatura en Partería Profesional (Mixta)
AREA: Ciencias de la Salud
UNIDAD ACADEMICA: Centro Regional de Educación Superior Taxco el Viejo Campus Zona Norte
NIVEL: Superior
MODALIDAD: Mixta
DURACION: 7 Semestres o 3.5 años.
LOCALIDAD: Taxco el viejo
REGION: Norte

--- Resultado 3 ---
Similitud: 0.564
UNIDAD ACADEMICA: Centro Regional de Educación Superior Taxco el Viejo Campus Zona Norte
PROGRAMA: Licenciatura en Fisiotera

In [14]:
# ============================================
# BLOQUE 6.3: PRUEBA CUAJINICUILAPA
# ============================================

resultados_cuajinicuilapa = buscar_fragmentos(
    "¿Qué carreras ofrece la UAGro en Cuajinicuilapa?",
    cantidad=8
)

for i, resultado in enumerate(resultados_cuajinicuilapa, start=1):
    print(f"\n--- Resultado {i} ---")
    print("Similitud:", round(resultado["similitud"], 3))
    print(resultado["documento"])


--- Resultado 1 ---
Similitud: 0.341
UNIDAD ACADEMICA: Facultad de Medicina Veterinaria y Zootecnia No. 2
PROGRAMA: Licenciatura en Medicina Veterinaria y Zootecnia
LOCALIDAD: Cuajinicuilapa
REGION: Costa Chica
DIRECTOR: M.C. María Benedicta Bottini Luzarlo
DIRECCION: 41949, Fed. Acapulco - Pinotepa Nacional 131, San Francisco, Cuajinicuilapa, Gro. 7. OFERTA EDUCATIVA COSTA GRANDE
TELEFONO: Tel. 747 471 93 10 Ext. 3629, 3628, 7414140783 (número central UAGro)
CORREO: fmvz2@uagro.mx
PAGINA WEB: www.fmvz2.uagro.mx
FACEBOOK: https://www.facebook.com/FMVZ2

--- Resultado 2 ---
Similitud: 0.289
CARRERA: Licenciatura en Medicina Veterinaria y Zootecnia
AREA: Biotecnología y Ciencias Agropecuarias
UNIDAD ACADEMICA: Facultad de Medicina Veterinaria y Zootecnia No. 2
NIVEL: Superior
MODALIDAD: Escolarizada
DURACION: 10 Semestres o 5 años
LOCALIDAD: Cuajinicuilapa
REGION: Costa Chica


In [15]:
# ============================================
# BLOQUE 7: ASISTENTE RAG + GROQ
# ============================================

def asistente(pregunta):

    resultados = buscar_fragmentos(
        pregunta,
        cantidad=8
    )

    if not resultados:
        return (
            "No encontré información suficiente en la base de datos "
            "para responder esa pregunta."
        )

    contexto = "\n\n".join(
        resultado["documento"]
        for resultado in resultados
    )

    prompt = f"""
Eres UAGro Orienta, un asistente académico de la
Universidad Autónoma de Guerrero.

Responde únicamente con información que aparezca en el contexto.
No inventes carreras, localidades, unidades académicas,
modalidades, duraciones, teléfonos, correos ni requisitos.
No mezcles información de distintas localidades.
No muestres palabras clave ni intenciones internas.

Si la información no aparece claramente en el contexto,
responde:

"No encontré información suficiente en la base de datos
para responder esa pregunta."

Responde en español, con claridad y usando listas cuando sea útil.
Completa todas las oraciones.

CONTEXTO:
{contexto}

PREGUNTA:
{pregunta}

RESPUESTA:
"""

    respuesta = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_tokens=400
    )

    return respuesta.choices[0].message.content.strip()

In [18]:
# ============================================
# BLOQUE 7.1: PRUEBAS DEL ASISTENTE
# ============================================

preguntas_prueba = [
    "¿Qué carreras ofrece la UAGro en Zihuatanejo?",
    "¿Qué carreras ofrece la UAGro en Cuajinicuilapa?",
    "¿Qué carreras de salud puedo estudiar en la Montaña?",
    "¿Dónde puedo estudiar Enfermería en Acapulco?",
    "¿Cuál es el contacto de la Escuela Superior de Enfermería No. 1?"
]

for pregunta in preguntas_prueba:
    print("=" * 80)
    print("PREGUNTA:", pregunta)
    print("RESPUESTA:")
    print(asistente(pregunta))
    print()

PREGUNTA: ¿Qué carreras ofrece la UAGro en Zihuatanejo?
RESPUESTA:
- Licenciatura en Innovación Hotelera y Gestión Turística Sustentable.

PREGUNTA: ¿Qué carreras ofrece la UAGro en Cuajinicuilapa?
RESPUESTA:
La UAGro ofrece la siguiente carrera en Cuajinicuilapa:

- Licenciatura en Medicina Veterinaria y Zootecnia.

PREGUNTA: ¿Qué carreras de salud puedo estudiar en la Montaña?
RESPUESTA:
**Carreras de salud disponibles en la región Montaña:**

- Licenciatura en Nutrición y Ciencia de Los Alimentos  
  - Área: Ciencias de la Salud  
  - Unidad académica: Centro Regional de Educación Superior de la Montaña Campus Huamuxtitlan  
  - Modalidad: Escolarizada  
  - Duración: 10 semestres (5 años)  
  - Localidad: Huamuxtitlán  
  - Región: Montaña

PREGUNTA: ¿Dónde puedo estudiar Enfermería en Acapulco?
RESPUESTA:
Para estudiar Licenciatura en Enfermería en Acapulco puedes acudir a la **Facultad de Enfermería No. 2**.  

**Datos de la unidad académica:**

- **Dirección:** P.º de la Cañada 